In [5]:
# ============================================
# TEMPLATE PREPROCESSING - REUSABLE
# ============================================

# Data manipulation and numerical computing
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Train/test split and scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Evaluation metrics
from sklearn.metrics import (classification_report, 
                             roc_auc_score, 
                             ConfusionMatrixDisplay)

# Decision Tree and Random Forest models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import AdaBoostClassifier

# --------------------------------------------
# 1. LOAD DATA
# --------------------------------------------
df = pd.read_csv(r"C:\Users\Giada Jenny Qafalia\Desktop\develhope\TEORIA\data-science-ml-portfolio\data\WA_Fn-UseC_-Telco-Customer-Churn.xls")

# --------------------------------------------
# 2. CLEANING
# --------------------------------------------
# TotalCharges has hidden spaces ' ' → convert to numeric
# errors='coerce' turns invalid values into NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop rows with NaN (only 11 rows affected)
df.dropna(inplace=True)

# Drop customer ID → not a predictive feature
df.drop(columns=['customerID'], inplace=True)

# Encode target variable as binary
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# --------------------------------------------
# 3. DEFINE FEATURES
# --------------------------------------------
# Continuous columns → will be scaled
continuous_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Categorical columns → will be one-hot encoded
# Automatically selects all columns except continuous and target
categorical_cols = [col for col in df.columns 
                    if col not in continuous_cols + ['Churn']]

# --------------------------------------------
# 4. ONE-HOT ENCODING
# --------------------------------------------
# drop_first=True avoids the dummy variable trap
# e.g. Gender: Male/Female → only Female column kept
df_encoded = pd.get_dummies(df, 
                             columns=categorical_cols, 
                             drop_first=True)

# --------------------------------------------
# 5. SPLIT X AND y
# --------------------------------------------
# X → all features (everything except target)
# y → target variable (Churn)
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# --------------------------------------------
# 6. TRAIN/TEST SPLIT
# --------------------------------------------
# test_size=0.2 → 80% train, 20% test
# stratify=y → preserves class balance in both sets
# random_state=42 → reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 7. ── SCALING — obbligatorio per K-Means ───────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Preprocessing complete")
print(f"   Dataset: {X.shape}")
print(f"   Churn rate: {df['Churn'].mean():.2%}")

✅ Preprocessing complete
   Dataset: (7032, 30)
   Churn rate: 26.58%


In [6]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Model definition
model = MLPClassifier(
    hidden_layer_sizes=(64, 32),  # two hidden layers
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42
)

# Training and evaluation
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.3f}")

              precision    recall  f1-score   support

           0       0.80      0.95      0.87      1033
           1       0.70      0.33      0.45       374

    accuracy                           0.79      1407
   macro avg       0.75      0.64      0.66      1407
weighted avg       0.77      0.79      0.76      1407

AUC-ROC: 0.754


## Valutazione dei Modelli — ANN vs Modelli Baseline

| Modello        | AUC-ROC | Recall (Churn) | F1   |
|----------------|---------|----------------|------|
| Logistic Reg   | 0.842   | 0.56           | 0.59 |
| KNN (K=27)     | 0.828   | 0.59           | 0.60 |
| Random Forest  | 0.832   | 0.40           | 0.50 |
| LDA            | 0.828   | 0.56           | 0.59 |
| Decision Tree  | 0.820   | 0.50           | 0.55 |
| SVM            | 0.783   | 0.48           | 0.55 |
| **ANN**        | **0.754**| **0.33**      | **0.45** |

## Osservazioni

Il modello ANN ha ottenuto le performance più basse su tutte le metriche.
Le reti neurali richiedono dataset di grandi dimensioni per apprendere 
configurazioni di pesi efficaci — con ~7.000 righe, il modello non dispone 
di dati sufficienti per generalizzare correttamente.

Il dataset è inoltre sbilanciato (73% No Churn / 27% Churn), il che penalizza 
il recall sulla classe minoritaria. Modelli più semplici come Logistic Regression 
e KNN ottengono risultati migliori perché i pattern nei dati tabulari di churn 
sono relativamente lineari e non richiedono estrazione profonda delle feature.

**Implicazione di business:** per dataset strutturati di piccole dimensioni 
(HR, CRM), i modelli interpretabili come Logistic Regression e KNN rimangono 
preferibili alle reti neurali sia in termini di performance che di spiegabilità.